# AIC 2026 — Post-Object Normalization → Retrieval-Ready Metadata

Notebook này xử lý riêng giai đoạn sau khi đã có:

- `processed-objects/keyframe_object_summary.parquet`
- `processed-objects/objects_index.parquet`

và dừng ngay trước bước Qwen embedding.

Output chính:

`MyDrive/data/retrieval-artifacts/master_keyframes.parquet`

In [1]:
!pip -q install -U pyarrow tqdm
print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.8 MB/s eta 0:00:00
Dependencies ready.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, re, unicodedata
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path("/content/drive/MyDrive/data")
PROCESSED_OBJECTS_ROOT = DATA_ROOT / "processed-objects"
RETRIEVAL_ROOT = DATA_ROOT / "retrieval-artifacts"
RETRIEVAL_ROOT.mkdir(parents=True, exist_ok=True)

MASTER_CSV = DATA_ROOT / "keyframe_master.csv"
OBJECT_SUMMARY_PATH = PROCESSED_OBJECTS_ROOT / "keyframe_object_summary.parquet"
OBJECT_INDEX_PATH = PROCESSED_OBJECTS_ROOT / "objects_index.parquet"
FINAL_MASTER_PATH = RETRIEVAL_ROOT / "master_keyframes.parquet"

ASR_PATH_CANDIDATES = [
    DATA_ROOT / "asr.parquet",
    DATA_ROOT / "processed-asr" / "asr.parquet",
    RETRIEVAL_ROOT / "asr.parquet",
]

OCR_CACHE_ROOT = DATA_ROOT / "ocr-cache"
OCR_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

TEMPORAL_GAP_SECONDS = 12.0
ASR_WINDOW_SECONDS = 8.0
MAX_METADATA_CHARS = 2000

assert MASTER_CSV.exists(), MASTER_CSV
assert OBJECT_SUMMARY_PATH.exists(), OBJECT_SUMMARY_PATH
assert OBJECT_INDEX_PATH.exists(), OBJECT_INDEX_PATH

Mounted at /content/drive


In [3]:
master = pd.read_csv(
    MASTER_CSV,
    dtype={"video_id": "string", "keyframe_id": "string"},
)
master = master.loc[:, ~master.columns.astype(str).str.startswith("Unnamed:")].copy()

object_summary = pd.read_parquet(OBJECT_SUMMARY_PATH)
object_summary["video_id"] = object_summary["video_id"].astype("string")
object_summary["keyframe_id"] = object_summary["keyframe_id"].astype("string")

print("master:", master.shape)
print("object_summary:", object_summary.shape)
display(master.head())
display(object_summary.head())

master: (177321, 9)
object_summary: (177321, 5)


,global_id,video_id,keyframe_id,n,pts_time,fps,frame_idx,image_path,map_mode
0,0,L21_V001,001,1,0.0000,30.0,0,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
1,1,L21_V001,002,2,3.0000,30.0,90,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
2,2,L21_V001,003,3,8.7000,30.0,261,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
3,3,L21_V001,004,4,11.7333,30.0,351,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
4,4,L21_V001,005,5,13.7000,30.0,411,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n


,global_id,video_id,keyframe_id,objects_json,has_raw_object_json
0,0,L21_V001,001,"{""lantern"": {""count"": 2, ""max_conf"": 0.7967387...",True
1,1,L21_V001,002,"{""boat"": {""count"": 2, ""max_conf"": 0.6481285, ""...",True
2,2,L21_V001,003,"{""clothing"": {""count"": 3, ""max_conf"": 0.659825...",True
3,3,L21_V001,004,"{""pumpkin"": {""count"": 1, ""max_conf"": 0.7350670...",True
4,4,L21_V001,005,"{""house"": {""count"": 7, ""max_conf"": 0.70072985,...",True


In [4]:
required_master_cols = {"global_id", "video_id", "keyframe_id"}
missing = required_master_cols - set(master.columns)
assert not missing, f"Missing master columns: {missing}"

master["global_id"] = pd.to_numeric(master["global_id"], errors="raise").astype("int64")
assert master["global_id"].is_unique
assert master[["video_id", "keyframe_id"]].duplicated().sum() == 0

master = master.sort_values("global_id").reset_index(drop=True)

print("Rows:", f"{len(master):,}")
print("Videos:", f"{master['video_id'].nunique():,}")

Rows: 177,321
Videos: 873


In [5]:
required_obj_cols = {"global_id", "objects_json", "has_raw_object_json"}
missing_obj = required_obj_cols - set(object_summary.columns)
assert not missing_obj, f"Missing object columns: {missing_obj}"

obj_summary_small = object_summary[
    ["global_id", "objects_json", "has_raw_object_json"]
].copy()

obj_summary_small["global_id"] = pd.to_numeric(
    obj_summary_small["global_id"], errors="raise"
).astype("int64")

assert obj_summary_small["global_id"].is_unique

master = master.merge(
    obj_summary_small,
    on="global_id",
    how="left",
    validate="one_to_one",
)

master["objects_json"] = master["objects_json"].fillna("{}")
master["has_raw_object_json"] = (
    master["has_raw_object_json"].fillna(False).astype(bool)
)

display(master.head())

,global_id,video_id,keyframe_id,n,pts_time,fps,frame_idx,image_path,map_mode,objects_json,has_raw_object_json
0,0,L21_V001,001,1,0.0000,30.0,0,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n,"{""lantern"": {""count"": 2, ""max_conf"": 0.7967387...",True
1,1,L21_V001,002,2,3.0000,30.0,90,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n,"{""boat"": {""count"": 2, ""max_conf"": 0.6481285, ""...",True
2,2,L21_V001,003,3,8.7000,30.0,261,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n,"{""clothing"": {""count"": 3, ""max_conf"": 0.659825...",True
3,3,L21_V001,004,4,11.7333,30.0,351,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n,"{""pumpkin"": {""count"": 1, ""max_conf"": 0.7350670...",True
4,4,L21_V001,005,5,13.7000,30.0,411,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n,"{""house"": {""count"": 7, ""max_conf"": 0.70072985,...",True


In [6]:
def parse_objects_json(value):
    if value is None:
        return {}
    if isinstance(value, dict):
        return value
    try:
        return json.loads(value)
    except Exception:
        return {}

def build_object_fields(objects_json):
    data = parse_objects_json(objects_json)
    labels, weighted_tokens, descriptions = [], [], []

    for label, info in data.items():
        label = str(label).strip().lower()
        if not label:
            continue

        try:
            count = int(info.get("count", 1))
        except Exception:
            count = 1

        labels.append(label)
        weighted_tokens.extend([label] * max(1, min(count, 5)))
        descriptions.append(f"{label} x{count}")

    return pd.Series({
        "object_labels": " ".join(labels),
        "object_search_text": " ".join(weighted_tokens),
        "object_description": "; ".join(descriptions),
        "num_object_classes": len(labels),
    })

object_fields = master["objects_json"].apply(build_object_fields)
master = pd.concat([master, object_fields], axis=1)

display(master[
    ["global_id", "object_labels", "object_search_text", "object_description"]
].head(10))

,global_id,object_labels,object_search_text,object_description
0,0,lantern skyscraper poster tower,lantern lantern skyscraper skyscraper skyscrap...,lantern x2; skyscraper x7; poster x1; tower x4
1,1,boat clothing person skyscraper woman human fa...,boat boat clothing clothing clothing clothing ...,boat x2; clothing x4; person x3; skyscraper x1...
2,2,clothing person man woman boat human face skys...,clothing clothing clothing person person man m...,clothing x3; person x2; man x2; woman x1; boat...
3,3,pumpkin clothing lantern,pumpkin clothing lantern,pumpkin x1; clothing x1; lantern x1
4,4,house tree plant porch,house house house house house tree tree tree t...,house x7; tree x9; plant x4; porch x1
5,5,vehicle airplane traffic sign,vehicle vehicle airplane traffic sign,vehicle x2; airplane x1; traffic sign x1
6,6,land vehicle vehicle human face car person clo...,land vehicle land vehicle land vehicle vehicle...,land vehicle x3; vehicle x1; human face x1; ca...
7,7,flag airplane,flag flag flag flag flag airplane,flag x5; airplane x1
8,8,human face man clothing woman skyscraper girl ...,human face human face man clothing clothing wo...,human face x2; man x1; clothing x2; woman x1; ...
9,9,human face man clothing woman skyscraper girl ...,human face human face man clothing clothing wo...,human face x2; man x1; clothing x2; woman x1; ...


In [7]:
_whitespace_re = re.compile(r"\s+")

def normalize_text(text):
    if text is None:
        return ""
    if isinstance(text, float) and np.isnan(text):
        return ""

    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\x00", " ")
    text = _whitespace_re.sub(" ", text)
    return text.strip()

TEXT_COLUMN_CANDIDATES = [
    "caption", "caption_text",
    "vlm_caption", "vlm_text",
    "description", "scene_description",
    "ocr_text", "asr_text", "transcript",
    "video_title", "video_name", "scene",
]

available_text_cols = [
    c for c in TEXT_COLUMN_CANDIDATES if c in master.columns
]

print("Detected textual columns:", available_text_cols)

for c in available_text_cols:
    master[c] = master[c].apply(normalize_text)

Detected textual columns: []


In [8]:
TIME_COLUMN_CANDIDATES = [
    "pts_time", "timestamp", "frame_time", "time_seconds"
]

TIME_COL = next(
    (c for c in TIME_COLUMN_CANDIDATES if c in master.columns),
    None
)

print("Temporal column:", TIME_COL)

if TIME_COL is not None:
    master[TIME_COL] = pd.to_numeric(master[TIME_COL], errors="coerce")
    display(master[TIME_COL].describe())

Temporal column: pts_time


,pts_time
count,177321.000000
mean,370.974716
std,408.785442
min,0.000000
25%,98.560000
50%,215.040000
75%,510.000000
max,2734.640000


In [9]:
def join_nonempty(values):
    values = [normalize_text(v) for v in values]
    return " ".join(v for v in values if v)

TEMPORAL_TEXT_FIELDS = [
    c for c in [
        "caption", "caption_text",
        "vlm_caption", "vlm_text",
        "scene_description", "asr_text", "transcript"
    ]
    if c in master.columns
]

if TEMPORAL_TEXT_FIELDS:
    master["frame_semantic_text"] = master[TEMPORAL_TEXT_FIELDS].apply(
        lambda r: join_nonempty(r.values), axis=1
    )
else:
    master["frame_semantic_text"] = ""

sort_cols = ["video_id", TIME_COL if TIME_COL is not None else "global_id"]
master = master.sort_values(sort_cols).reset_index(drop=True)

grouped = master.groupby("video_id", sort=False)
master["prev_frame_text"] = grouped["frame_semantic_text"].shift(1).fillna("")
master["next_frame_text"] = grouped["frame_semantic_text"].shift(-1).fillna("")

if TIME_COL is not None:
    master["prev_pts_time"] = grouped[TIME_COL].shift(1)
    master["next_pts_time"] = grouped[TIME_COL].shift(-1)

    prev_gap = master[TIME_COL] - master["prev_pts_time"]
    next_gap = master["next_pts_time"] - master[TIME_COL]

    master.loc[prev_gap > TEMPORAL_GAP_SECONDS, "prev_frame_text"] = ""
    master.loc[next_gap > TEMPORAL_GAP_SECONDS, "next_frame_text"] = ""

master["temporal_context_text"] = master[
    ["prev_frame_text", "frame_semantic_text", "next_frame_text"]
].apply(lambda r: join_nonempty(r.values), axis=1)

In [10]:
ASR_PATH = next((p for p in ASR_PATH_CANDIDATES if p.exists()), None)
print("ASR_PATH:", ASR_PATH)

asr = pd.read_parquet(ASR_PATH) if ASR_PATH is not None else None

if asr is not None:
    print("ASR columns:", asr.columns.tolist())
    display(asr.head())

ASR_PATH: None


In [11]:
def attach_asr_window(master_df, asr_df, time_col, window_seconds=8.0):
    result = np.empty(len(master_df), dtype=object)
    result[:] = ""

    asr_df = asr_df.copy()
    required = {"video_id", "start", "end", "text"}
    missing = required - set(asr_df.columns)
    if missing:
        raise ValueError(f"ASR missing columns: {missing}")

    asr_df["video_id"] = asr_df["video_id"].astype(str)
    asr_df["start"] = pd.to_numeric(asr_df["start"], errors="coerce")
    asr_df["end"] = pd.to_numeric(asr_df["end"], errors="coerce")
    asr_df["text"] = asr_df["text"].apply(normalize_text)

    grouped_asr = {
        vid: g.sort_values("start")
        for vid, g in asr_df.groupby("video_id")
    }

    for i, row in tqdm(
        enumerate(master_df.itertuples(index=False)),
        total=len(master_df),
        desc="Attach ASR",
    ):
        vid = str(row.video_id)
        if vid not in grouped_asr:
            continue

        t = getattr(row, time_col)
        if pd.isna(t):
            continue

        left = float(t) - window_seconds
        right = float(t) + window_seconds

        g = grouped_asr[vid]
        matched = g[(g["end"] >= left) & (g["start"] <= right)]

        if len(matched):
            result[i] = " ".join(matched["text"].dropna().tolist())

    return result

if asr is not None and TIME_COL is not None:
    master["asr_context_text"] = attach_asr_window(
        master,
        asr,
        time_col=TIME_COL,
        window_seconds=ASR_WINDOW_SECONDS,
    )
else:
    master["asr_context_text"] = ""

In [12]:
if "ocr_text" in master.columns:
    master["ocr_text"] = master["ocr_text"].apply(normalize_text)
else:
    master["ocr_text"] = ""

print("OCR cache root:", OCR_CACHE_ROOT)

OCR cache root: /content/drive/MyDrive/data/ocr-cache


In [13]:
LEXICAL_FIELDS = [
    c for c in [
        "caption", "caption_text",
        "vlm_caption", "vlm_text",
        "scene_description",
        "asr_context_text",
        "ocr_text",
        "object_search_text",
        "video_title", "video_name",
    ]
    if c in master.columns
]

print("BM25 fields:", LEXICAL_FIELDS)

master["lexical_text"] = (
    master[LEXICAL_FIELDS].apply(lambda r: join_nonempty(r.values), axis=1)
    if LEXICAL_FIELDS else ""
)

BM25 fields: ['asr_context_text', 'ocr_text', 'object_search_text']


In [14]:
SEMANTIC_FIELDS = [
    c for c in [
        "frame_semantic_text",
        "object_description",
        "asr_context_text",
        "ocr_text",
    ]
    if c in master.columns
]

master["semantic_metadata_text"] = (
    master[SEMANTIC_FIELDS].apply(lambda r: join_nonempty(r.values), axis=1)
    if SEMANTIC_FIELDS else ""
)

def truncate_text(text, max_chars=MAX_METADATA_CHARS):
    text = normalize_text(text)
    return text if len(text) <= max_chars else text[:max_chars].rstrip()

for col in [
    "lexical_text",
    "semantic_metadata_text",
    "temporal_context_text",
    "asr_context_text",
]:
    master[col] = master[col].apply(truncate_text)

In [15]:
sort_cols = ["video_id", TIME_COL if TIME_COL is not None else "global_id"]
master = master.sort_values(sort_cols).reset_index(drop=True)

g = master.groupby("video_id", sort=False)

master["prev_global_id"] = g["global_id"].shift(1).astype("Int64")
master["next_global_id"] = g["global_id"].shift(-1).astype("Int64")

In [16]:
master["temporal_segment_id"] = -1
segment_counter = 0

for video_id, idx in tqdm(
    master.groupby("video_id", sort=False).groups.items(),
    desc="Temporal segments",
):
    idx = list(idx)

    if TIME_COL is None:
        master.loc[idx, "temporal_segment_id"] = np.arange(
            segment_counter,
            segment_counter + len(idx),
        )
        segment_counter += len(idx)
        continue

    times = master.loc[idx, TIME_COL].to_numpy()
    current_segment = segment_counter
    previous_t = None

    for row_idx, t in zip(idx, times):
        if (
            previous_t is not None
            and not pd.isna(t)
            and not pd.isna(previous_t)
            and t - previous_t > TEMPORAL_GAP_SECONDS
        ):
            current_segment += 1

        master.at[row_idx, "temporal_segment_id"] = current_segment
        previous_t = t

    segment_counter = current_segment + 1

master["temporal_segment_id"] = master["temporal_segment_id"].astype("int64")
print("Temporal segments:", f"{master['temporal_segment_id'].nunique():,}")

Temporal segments:   0%|          | 0/873 [00:00<?, ?it/s]

Temporal segments: 873


In [17]:
def nonempty_series(s):
    return s.fillna("").astype(str).str.strip().ne("")

master["has_objects"] = nonempty_series(master["object_labels"])
master["has_ocr"] = nonempty_series(master["ocr_text"])
master["has_asr"] = nonempty_series(master["asr_context_text"])
master["has_semantic_text"] = nonempty_series(master["frame_semantic_text"])

master["metadata_coverage_score"] = (
    master["has_objects"].astype(int)
    + master["has_ocr"].astype(int)
    + master["has_asr"].astype(int)
    + master["has_semantic_text"].astype(int)
)

coverage_cols = [
    "has_raw_object_json",
    "has_objects",
    "has_ocr",
    "has_asr",
    "has_semantic_text",
]

coverage_df = pd.DataFrame([
    {
        "field": c,
        "count": int(master[c].sum()),
        "coverage": float(master[c].mean()),
    }
    for c in coverage_cols
])

display(coverage_df)

,field,count,coverage
0,has_raw_object_json,177321,1.000000
1,has_objects,171741,0.968532
2,has_ocr,0,0.000000
3,has_asr,0,0.000000
4,has_semantic_text,0,0.000000


In [18]:
objects_index = pd.read_parquet(
    OBJECT_INDEX_PATH,
    columns=["label", "confidence", "area_norm"],
)

print("Total detections:", f"{len(objects_index):,}")

display(
    objects_index["label"]
    .value_counts()
    .head(30)
    .to_frame("count")
)

display(objects_index[["confidence", "area_norm"]].describe())

Total detections: 1,622,347


,count
label,
Person,223882
Clothing,200317
Human face,128646
Food,106327
Man,100224
Tree,86633
Woman,69683
Plant,36715
Table,30696


,confidence,area_norm
count,1.622347e+06,1.622347e+06
mean,4.578626e-01,1.551837e-01
std,2.097119e-01,2.078685e-01
min,2.000005e-01,9.590298e-05
25%,2.759250e-01,1.340882e-02
50%,4.026088e-01,6.281403e-02
75%,6.108506e-01,2.121350e-01
max,9.994711e-01,1.000000e+00


In [19]:
empty_lexical = master["lexical_text"].fillna("").str.strip().eq("")

print(
    "Empty lexical docs:",
    f"{empty_lexical.sum():,}",
    "/",
    f"{len(master):,}",
    f"({empty_lexical.mean():.2%})",
)

Empty lexical docs: 5,580 / 177,321 (3.15%)


In [20]:
preferred_columns = [
    "global_id",
    "video_id",
    "keyframe_id",
]

if TIME_COL is not None:
    preferred_columns.append(TIME_COL)

preferred_columns += [
    "image_path",
    "objects_json",
    "object_labels",
    "object_description",
    "object_search_text",
    "num_object_classes",
    "frame_semantic_text",
    "asr_context_text",
    "ocr_text",
    "lexical_text",
    "semantic_metadata_text",
    "temporal_context_text",
    "prev_global_id",
    "next_global_id",
    "temporal_segment_id",
    "has_raw_object_json",
    "has_objects",
    "has_ocr",
    "has_asr",
    "has_semantic_text",
    "metadata_coverage_score",
]

preferred_columns = [c for c in preferred_columns if c in master.columns]
remaining_columns = [c for c in master.columns if c not in preferred_columns]

master = master[preferred_columns + remaining_columns]

print("Final columns:", len(master.columns))
print(master.columns.tolist())

Final columns: 33
['global_id', 'video_id', 'keyframe_id', 'pts_time', 'image_path', 'objects_json', 'object_labels', 'object_description', 'object_search_text', 'num_object_classes', 'frame_semantic_text', 'asr_context_text', 'ocr_text', 'lexical_text', 'semantic_metadata_text', 'temporal_context_text', 'prev_global_id', 'next_global_id', 'temporal_segment_id', 'has_raw_object_json', 'has_objects', 'has_ocr', 'has_asr', 'has_semantic_text', 'metadata_coverage_score', 'n', 'fps', 'frame_idx', 'map_mode', 'prev_frame_text', 'next_frame_text', 'prev_pts_time', 'next_pts_time']


In [21]:
assert master["global_id"].is_unique
assert master[["video_id", "keyframe_id"]].duplicated().sum() == 0
assert master["objects_json"].notna().all()
assert master["lexical_text"].notna().all()
assert master["semantic_metadata_text"].notna().all()
assert master["temporal_context_text"].notna().all()

print("Rows:", f"{len(master):,}")
print("Videos:", f"{master['video_id'].nunique():,}")
print("Object coverage:", f"{master['has_objects'].mean():.2%}")
print("ASR coverage:", f"{master['has_asr'].mean():.2%}")
print("OCR coverage:", f"{master['has_ocr'].mean():.2%}")
print("Semantic-text coverage:", f"{master['has_semantic_text'].mean():.2%}")

display(master.head())

Rows: 177,321
Videos: 873
Object coverage: 96.85%
ASR coverage: 0.00%
OCR coverage: 0.00%
Semantic-text coverage: 0.00%


,global_id,video_id,keyframe_id,pts_time,image_path,objects_json,object_labels,object_description,object_search_text,num_object_classes,...,has_semantic_text,metadata_coverage_score,n,fps,frame_idx,map_mode,prev_frame_text,next_frame_text,prev_pts_time,next_pts_time
0,0,L21_V001,001,0.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""lantern"": {""count"": 2, ""max_conf"": 0.7967387...",lantern skyscraper poster tower,lantern x2; skyscraper x7; poster x1; tower x4,lantern lantern skyscraper skyscraper skyscrap...,4,...,False,1,1,30.0,0,direct_n,,,NaN,3.0000
1,1,L21_V001,002,3.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""boat"": {""count"": 2, ""max_conf"": 0.6481285, ""...",boat clothing person skyscraper woman human fa...,boat x2; clothing x4; person x3; skyscraper x1...,boat boat clothing clothing clothing clothing ...,9,...,False,1,2,30.0,90,direct_n,,,0.0000,8.7000
2,2,L21_V001,003,8.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""clothing"": {""count"": 3, ""max_conf"": 0.659825...",clothing person man woman boat human face skys...,clothing x3; person x2; man x2; woman x1; boat...,clothing clothing clothing person person man m...,9,...,False,1,3,30.0,261,direct_n,,,3.0000,11.7333
3,3,L21_V001,004,11.7333,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""pumpkin"": {""count"": 1, ""max_conf"": 0.7350670...",pumpkin clothing lantern,pumpkin x1; clothing x1; lantern x1,pumpkin clothing lantern,3,...,False,1,4,30.0,351,direct_n,,,8.7000,13.7000
4,4,L21_V001,005,13.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""house"": {""count"": 7, ""max_conf"": 0.70072985,...",house tree plant porch,house x7; tree x9; plant x4; porch x1,house house house house house tree tree tree t...,4,...,False,1,5,30.0,411,direct_n,,,11.7333,17.7000


In [22]:
master = master.sort_values("global_id").reset_index(drop=True)

master.to_parquet(
    FINAL_MASTER_PATH,
    index=False,
    compression="zstd",
)

print("Saved:", FINAL_MASTER_PATH)
print("Size:", f"{FINAL_MASTER_PATH.stat().st_size / 1024**2:.2f} MB")

Saved: /content/drive/MyDrive/data/retrieval-artifacts/master_keyframes.parquet
Size: 27.47 MB


In [23]:
master_check = pd.read_parquet(FINAL_MASTER_PATH)

assert len(master_check) == len(master)
assert master_check["global_id"].is_unique
assert master_check["global_id"].tolist() == master["global_id"].tolist()

print("master_keyframes.parquet verified.")
print("Rows:", f"{len(master_check):,}")
print("Columns:", len(master_check.columns))

display(master_check.head())

print("\nPOST-OBJECT → PRE-QWEN PREPROCESSING: COMPLETE")

master_keyframes.parquet verified.
Rows: 177,321
Columns: 33


,global_id,video_id,keyframe_id,pts_time,image_path,objects_json,object_labels,object_description,object_search_text,num_object_classes,...,has_semantic_text,metadata_coverage_score,n,fps,frame_idx,map_mode,prev_frame_text,next_frame_text,prev_pts_time,next_pts_time
0,0,L21_V001,001,0.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""lantern"": {""count"": 2, ""max_conf"": 0.7967387...",lantern skyscraper poster tower,lantern x2; skyscraper x7; poster x1; tower x4,lantern lantern skyscraper skyscraper skyscrap...,4,...,False,1,1,30.0,0,direct_n,,,NaN,3.0000
1,1,L21_V001,002,3.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""boat"": {""count"": 2, ""max_conf"": 0.6481285, ""...",boat clothing person skyscraper woman human fa...,boat x2; clothing x4; person x3; skyscraper x1...,boat boat clothing clothing clothing clothing ...,9,...,False,1,2,30.0,90,direct_n,,,0.0000,8.7000
2,2,L21_V001,003,8.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""clothing"": {""count"": 3, ""max_conf"": 0.659825...",clothing person man woman boat human face skys...,clothing x3; person x2; man x2; woman x1; boat...,clothing clothing clothing person person man m...,9,...,False,1,3,30.0,261,direct_n,,,3.0000,11.7333
3,3,L21_V001,004,11.7333,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""pumpkin"": {""count"": 1, ""max_conf"": 0.7350670...",pumpkin clothing lantern,pumpkin x1; clothing x1; lantern x1,pumpkin clothing lantern,3,...,False,1,4,30.0,351,direct_n,,,8.7000,13.7000
4,4,L21_V001,005,13.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""house"": {""count"": 7, ""max_conf"": 0.70072985,...",house tree plant porch,house x7; tree x9; plant x4; porch x1,house house house house house tree tree tree t...,4,...,False,1,5,30.0,411,direct_n,,,11.7333,17.7000



POST-OBJECT → PRE-QWEN PREPROCESSING: COMPLETE


## Next step

Notebook tiếp theo mới thực hiện:

```text
master_keyframes.parquet
        ↓
Qwen3-VL-Embedding-2B
        ↓
1024-d embeddings
        ↓
FAISS
```

Các field như `lexical_text`, `objects_json`, `object_search_text`,
`asr_context_text`, `ocr_text` và `temporal_segment_id` được giữ riêng để
phục vụ BM25, RRF, verification, temporal grouping và reranking.